# BoostFL label-flipping robustness (BoT-IoT)

## 1. Imports

In [1]:
import os
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset

import flwr as fl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

warnings.filterwarnings("ignore")

## 2. Configuration

In [2]:
CSV_PATH = r"../../../data/Bot-IoT.csv"
TARGET_MULTICLASS = "category"
NORMAL_CLASS = "Normal"
DROP_COLS = ['attack', 'category', 'subcategory ', 'pkSeqID', 'saddr', 'daddr', 'soui', 'doui', 'sco', 'dco', 'smac', 'dmac']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NUM_CLIENTS = 10
NUM_PARTITIONS = 10
BATCH_SIZE = 32
EPOCHS = 5
EPSILON = 1e-8
LEARNING_RATE = 0.001
MAX_ALPHA = 10.0
MIN_ALPHA = 0.1
learning_rate_server = 1.0

ENABLE_ALPHA_FILTERING = True

BINARY = False
IID = False
DIRICHLET_ALPHA = 0.3

BASE_SEED = 2024
NUM_ROUNDS = 15

GPU_PER_CLIENT = 0.5 if torch.cuda.is_available() else 0.0
CPUS_PER_CLIENT = max(1, (os.cpu_count() or 2) // 2)

ENABLE_LABEL_FLIP = False
MALICIOUS_FRAC = 0.0
FLIP_PROB = 0.0
FLIP_MODE = "random"
SOURCE_CLASS = 0
TARGET_CLASS = 1
POISON_SEED = 2024
MALICIOUS_CLIENTS = set()

Using device: cuda


## 3. Data loading

In [3]:
def load_dataset(file_path, target_multiclass, normal_class, binary,
                 drop_cols, test_size=0.3, random_state=42):
    df = pd.read_csv(file_path)
    df = df.drop_duplicates()

    df = df.dropna(subset=[target_multiclass])
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    for col in categorical_cols:
        if df[col].isnull().any():
            mode_val = df[col].mode()
            df[col] = df[col].fillna(mode_val[0] if not mode_val.empty else "Unknown")

    y_multi = df[target_multiclass].astype(str).str.strip()
    if binary:
        y = np.where(y_multi.str.lower() == normal_class.lower(), "Benign", "Attack")
        y = pd.Series(y, index=df.index)
    else:
        y = y_multi

    X = df.drop(columns=drop_cols, errors="ignore").copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y)

    non_numeric_cols = list(
        set(X_train.select_dtypes(exclude=[np.number]).columns.tolist())
        | set(X_test.select_dtypes(exclude=[np.number]).columns.tolist()))
    feature_encoders = {}
    for col in non_numeric_cols:
        le_col = LabelEncoder()
        le_col.fit(X_train[col].astype(str))
        feature_encoders[col] = le_col
        mapping = {cls: idx for idx, cls in enumerate(le_col.classes_)}
        X_train[col] = le_col.transform(X_train[col].astype(str))
        X_test[col] = X_test[col].astype(str).map(mapping).fillna(-1).astype(int)

    def safe_numeric(df_):
        df_ = df_.apply(lambda c: c.map(lambda v: str(v).strip() if isinstance(v, str) else v))
        df_ = df_.apply(pd.to_numeric, errors="coerce")
        return df_.replace([np.inf, -np.inf], np.nan).fillna(0)

    X_train = safe_numeric(X_train)
    X_test = safe_numeric(X_test)

    global INPUT_DIM
    INPUT_DIM = X_train.shape[1]

    y_train = pd.Series(np.asarray(y_train)).astype(str).str.strip()
    y_test = pd.Series(np.asarray(y_test)).astype(str).str.strip()
    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train.values)
    y_test_enc = label_encoder.transform(y_test.values)
    class_names = label_encoder.classes_
    num_classes = len(class_names)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train.values.astype(np.float64))
    X_test_scaled = scaler.transform(X_test.values.astype(np.float64))

    train_dataset = TensorDataset(torch.from_numpy(X_train_scaled).float(),
                                  torch.from_numpy(y_train_enc).long())
    test_dataset = TensorDataset(torch.from_numpy(X_test_scaled).float(),
                                 torch.from_numpy(y_test_enc).long())
    print(f"Classes ({num_classes}): {list(class_names)}")
    print(f"Features: {INPUT_DIM} | Train: {len(train_dataset)} | Test: {len(test_dataset)}")
    return (train_dataset, test_dataset, class_names, num_classes,
            scaler, label_encoder, feature_encoders)


(
    train_dataset, test_dataset, class_names, NUM_CLASSES,
    scaler, label_encoder, feature_encoders,
) = load_dataset(CSV_PATH, TARGET_MULTICLASS, NORMAL_CLASS, BINARY, DROP_COLS)

Classes (4): ['DDoS/DoS', 'Normal', 'Reconnaissance', 'Theft']
Features: 23 | Train: 35791 | Test: 15339


## 4. Evaluation history

In [4]:
eval_loss_history = []
eval_accuracy_history = []
eval_precision_history = []
eval_recall_history = []
eval_f1_history = []
eval_grad_divergence_history = []
eval_rounds = []

## 5. Partitioning (IID and Non-IID)

In [5]:
def partition_dataset_iid(dataset, num_partitions):
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    indices_by_class = [[] for _ in range(NUM_CLASSES)]
    for idx, label in enumerate(labels):
        indices_by_class[label].append(idx)
    partitions = [[] for _ in range(num_partitions)]
    for c in range(NUM_CLASSES):
        indices = indices_by_class[c]
        np.random.shuffle(indices)
        per = len(indices) // num_partitions
        rem = len(indices) % num_partitions
        start = 0
        for p in range(num_partitions):
            extra = 1 if p < rem else 0
            end = start + per + extra
            partitions[p].extend(indices[start:end])
            start = end
    for p in range(num_partitions):
        np.random.shuffle(partitions[p])
    return partitions


def partition_dataset_dirichlet(dataset, num_partitions, dirichlet_alpha):
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    indices_by_class = [[] for _ in range(NUM_CLASSES)]
    for idx, label in enumerate(labels):
        indices_by_class[label].append(idx)
    partitions = [[] for _ in range(num_partitions)]
    for c in range(NUM_CLASSES):
        indices = indices_by_class[c]
        np.random.shuffle(indices)
        proportions = np.random.dirichlet([dirichlet_alpha] * num_partitions)
        counts = (proportions * len(indices)).astype(int)
        diff = len(indices) - counts.sum()
        if diff > 0:
            for k in np.argsort(proportions)[-diff:]:
                counts[k] += 1
        elif diff < 0:
            for k in np.argsort(proportions)[:abs(diff)]:
                if counts[k] > 0:
                    counts[k] -= 1
        start = 0
        for p in range(num_partitions):
            end = start + counts[p]
            partitions[p].extend(indices[start:end])
            start = end
    for p in range(num_partitions):
        np.random.shuffle(partitions[p])
    return partitions


def partition_dataset(dataset, num_partitions):
    if IID:
        return partition_dataset_iid(dataset, num_partitions)
    return partition_dataset_dirichlet(dataset, num_partitions, DIRICHLET_ALPHA)


train_partitions = partition_dataset(train_dataset, NUM_PARTITIONS)
print(f"Created {len(train_partitions)} partitions ({'IID' if IID else 'Non-IID'})")

Created 10 partitions (Non-IID)


## 6. Model, ensemble, and residual loss

In [6]:
class model(nn.Module):
    def __init__(self, INPUT_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_DIM, 50)
        self.fc2 = nn.Linear(50, 25)
        self.fc3 = nn.Linear(25, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class BoostFLEnsemble:
    def __init__(self, f0, device):
        self.f0 = f0.to(device)
        self.base_learners = []
        self.alphas = []
        self.device = device

    def add_learner(self, model_params, alpha):
        new_learner = model(INPUT_DIM, NUM_CLASSES).to(self.device)
        state_dict = new_learner.state_dict()
        new_state_dict = {
            k: torch.tensor(v).to(self.device) if isinstance(v, np.ndarray) else v.to(self.device)
            for k, v in zip(state_dict.keys(), model_params)}
        new_learner.load_state_dict(new_state_dict)
        new_learner.eval()
        self.base_learners.append(new_learner)
        self.alphas.append(alpha)

    def predict(self, x):
        with torch.no_grad():
            out = self.f0.unsqueeze(0).expand(x.size(0), -1).clone()
            if self.base_learners and self.alphas:
                total_alpha = sum(self.alphas)
                if total_alpha > EPSILON:
                    weighted_sum = torch.zeros_like(out)
                    for m, alpha in zip(self.base_learners, self.alphas):
                        m.eval()
                        weighted_sum += (alpha / total_alpha) * m(x)
                    out = out + weighted_sum
            return out

    def get_ensemble_info(self):
        return {"num_learners": len(self.base_learners),
                "alphas": self.alphas,
                "total_alpha": sum(self.alphas) if self.alphas else 0}


class ResidualLoss(nn.Module):
    def forward(self, predictions, residuals):
        return F.smooth_l1_loss(predictions, residuals)


class ClientEnsemble:
    def __init__(self, base_learners, alphas, f0):
        self.base_learners = base_learners
        self.alphas = alphas
        self.f0 = f0

    def predict(self, x):
        with torch.no_grad():
            out = self.f0.unsqueeze(0).expand(x.size(0), -1).clone()
            if self.base_learners and self.alphas:
                total_alpha = sum(self.alphas)
                if total_alpha > EPSILON:
                    weighted_sum = torch.zeros_like(out)
                    for m, alpha in zip(self.base_learners, self.alphas):
                        m.eval()
                        weighted_sum += (alpha / total_alpha) * m(x)
                    out = out + weighted_sum
            return out

## 7. Label-flipping wrapper

In [7]:
class LabelFlippedDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, num_classes, flip_prob=1.0, mode="random",
                 source_class=0, target_class=1, seed=0):
        self.base = base_dataset
        self.num_classes = int(num_classes)
        self.flip_prob = float(flip_prob)
        self.mode = str(mode)
        self.source_class = int(source_class)
        self.target_class = int(target_class)
        self.rng = np.random.RandomState(seed)

    def __len__(self):
        return len(self.base)

    def _flip_label(self, y):
        if self.mode == "targeted":
            return self.target_class if y == self.source_class else y
        new_y = y
        while new_y == y:
            new_y = int(self.rng.randint(0, self.num_classes))
        return new_y

    def __getitem__(self, idx):
        x, y = self.base[idx]
        y_int = int(y.item()) if torch.is_tensor(y) else int(y)
        if self.rng.rand() < self.flip_prob:
            y_int = self._flip_label(y_int)
        return x, torch.tensor(y_int, dtype=torch.long)

## 8. Flower client

In [8]:
def client_fn(cid):
    cid_int = int(cid)
    partition_indices = train_partitions[cid_int]
    base_client_subset = Subset(train_dataset, partition_indices)

    is_malicious = (ENABLE_LABEL_FLIP and (cid_int in MALICIOUS_CLIENTS))
    if is_malicious:
        client_dataset = LabelFlippedDataset(
            base_dataset=base_client_subset, num_classes=NUM_CLASSES,
            flip_prob=FLIP_PROB, mode=FLIP_MODE,
            source_class=SOURCE_CLASS, target_class=TARGET_CLASS,
            seed=POISON_SEED + cid_int)
    else:
        client_dataset = base_client_subset

    train_dataloader = DataLoader(client_dataset, batch_size=BATCH_SIZE, shuffle=True)

    f_t = model(INPUT_DIM, NUM_CLASSES).to(device)
    f0 = torch.randn(NUM_CLASSES, device=device) * 0.01
    residual_loss_fn = ResidualLoss()
    classification_loss_fn = nn.CrossEntropyLoss()
    learners = []
    alphas = []

    class FlowerClient(fl.client.NumPyClient):
        def __init__(self):
            self.learners = learners
            self.alphas = alphas
            self.f0 = f0
            self.f_t = f_t
            self.device = device
            self.train_dataset = client_dataset
            self.residual_loss_fn = residual_loss_fn
            self.classification_loss_fn = classification_loss_fn
            self.is_malicious = is_malicious

        def get_parameters(self, config=None):
            return [val.cpu().numpy() for val in self.f_t.state_dict().values()]

        def fit(self, parameters, config):
            state_dict = self.f_t.state_dict()
            new_state_dict = {k: torch.tensor(v).to(self.device)
                              for k, v in zip(state_dict.keys(), parameters)}
            self.f_t.load_state_dict(new_state_dict)
            self.f_t.train()

            optimizer = optim.Adam(self.f_t.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
            ensemble = ClientEnsemble(self.learners, self.alphas, self.f0)

            total_residual_loss = 0.0
            for epoch in range(EPOCHS):
                epoch_residual_loss = 0.0
                num_batches = 0
                for x_batch, y_batch in train_dataloader:
                    x_batch = x_batch.to(self.device)
                    y_batch = y_batch.to(self.device)
                    with torch.no_grad():
                        ensemble_logits = ensemble.predict(x_batch)
                        ensemble_probs = torch.softmax(ensemble_logits, dim=1)
                        y_one_hot = torch.zeros(y_batch.size(0), NUM_CLASSES, device=self.device)
                        y_one_hot.scatter_(1, y_batch.unsqueeze(1), 1)
                        residuals = y_one_hot - ensemble_probs
                    optimizer.zero_grad()
                    weak_learner_logits = self.f_t(x_batch)
                    weak_learner_probs = torch.softmax(weak_learner_logits, dim=1)
                    loss = self.residual_loss_fn(weak_learner_probs, residuals)
                    loss.backward()
                    
                    optimizer.step()
                    epoch_residual_loss += loss.item()
                    num_batches += 1
                total_residual_loss += epoch_residual_loss

            avg_residual_loss = (total_residual_loss / (EPOCHS * len(train_dataloader))
                                 if len(train_dataloader) > 0 else 1.0)
            avg_residual_loss = max(avg_residual_loss, EPSILON)
            avg_residual_loss = min(avg_residual_loss, 10.0)
            alpha_t = 1.0 / (1.0 + avg_residual_loss)
            alpha_t = max(MIN_ALPHA, min(MAX_ALPHA, alpha_t))

            new_learner = model(INPUT_DIM, NUM_CLASSES).to(device)
            new_learner.load_state_dict(self.f_t.state_dict())
            self.learners.append(new_learner)
            self.alphas.append(alpha_t)

            global_params = [torch.tensor(p).to(self.device) for p in parameters]
            local_params = list(self.f_t.state_dict().values())
            grad_divergence = sum((lp - gp).norm().item()
                                  for lp, gp in zip(local_params, global_params))

            return [val.cpu().numpy() for val in self.f_t.state_dict().values()], \
                len(self.train_dataset), {
                    "grad_divergence": grad_divergence,
                    "residual_loss": avg_residual_loss,
                    "alpha": alpha_t,
                    "is_malicious": int(is_malicious),
                }

        def evaluate(self, parameters, config):
            return 0.0, len(self.train_dataset), {"loss": 0.0}

    return FlowerClient().to_client()

## 9. Boosting strategy with alpha filtering

In [9]:
class BoostingStrategy(fl.server.strategy.FedAvg):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.current_global_params = None
        self.round_count = 0
        self.global_f0 = torch.randn(NUM_CLASSES, device=device) * 0.01
        self.global_ensemble = BoostFLEnsemble(self.global_f0, device)
        self.test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)
        self.classification_loss_fn = nn.CrossEntropyLoss()
        self.global_learners_history = []
        self.global_alphas_history = []
        self.mal_alpha_rounds = []
        self.mal_alpha_lists = []
        self.mal_alpha_means = []
        self.benign_alpha_lists = []
        self.benign_alpha_means = []
        self.num_filtered_per_round = []
        self.num_filtered_malicious_per_round = []
        self.alpha_threshold_per_round = []

    def initialize_parameters(self, client_manager):
        initial_model = model(INPUT_DIM, NUM_CLASSES).to(device)
        initial_params = [val.cpu().numpy() for val in initial_model.state_dict().values()]
        self.current_global_params = fl.common.ndarrays_to_parameters(initial_params)
        return self.current_global_params

    def evaluate_ensemble(self):
        y_true, y_pred = [], []
        total_loss = 0.0
        total_samples = 0
        with torch.no_grad():
            for x_batch, y_batch in self.test_dataloader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                ensemble_logits = self.global_ensemble.predict(x_batch)
                loss = self.classification_loss_fn(ensemble_logits, y_batch)
                total_loss += loss.item() * y_batch.size(0)
                total_samples += y_batch.size(0)
                _, preds = torch.max(ensemble_logits, 1)
                y_true.extend(y_batch.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())
        return {
            "loss": total_loss / total_samples if total_samples > 0 else 0.0,
            "accuracy": accuracy_score(y_true, y_pred) if y_true else 0.0,
            "precision": precision_score(y_true, y_pred, average="macro", zero_division=0) if y_true else 0.0,
            "recall": recall_score(y_true, y_pred, average="macro", zero_division=0) if y_true else 0.0,
            "f1_score": f1_score(y_true, y_pred, average="macro", zero_division=0) if y_true else 0.0,
        }

    def aggregate_fit(self, rnd, results, failures):
        print(f"[Round {rnd}] {len(results)} clients succeeded, {len(failures)} failed")
        self.round_count = rnd
        if not results:
            return self.current_global_params, {}

        if self.current_global_params is None:
            initial_model = model(INPUT_DIM, NUM_CLASSES).to(device)
            initial_params = [val.cpu().numpy() for val in initial_model.state_dict().values()]
            self.current_global_params = fl.common.ndarrays_to_parameters(initial_params)

        old_global_params = fl.common.parameters_to_ndarrays(self.current_global_params)
        weighted_updates = [np.zeros_like(p) for p in old_global_params]
        sum_alpha = 0.0
        num_examples_total = 0
        residual_losses = []
        alphas = []
        grad_divergences = []
        valid_results = []

        for client_res in results:
            try:
                if isinstance(client_res, tuple):
                    if len(client_res) == 3:
                        parameters, num_examples, metrics = client_res
                    elif len(client_res) == 2:
                        _, fit_res = client_res
                        parameters = fit_res.parameters
                        num_examples = fit_res.num_examples
                        metrics = fit_res.metrics
                    else:
                        continue
                else:
                    parameters = client_res.parameters
                    num_examples = getattr(client_res, "num_examples", 0)
                    metrics = getattr(client_res, "metrics", {})
                if not isinstance(metrics, dict) and hasattr(metrics, "metrics"):
                    metrics = metrics.metrics

                alpha = float(metrics.get("alpha", 1.0))
                if math.isnan(alpha) or math.isinf(alpha) or alpha <= 0:
                    alpha = 1.0
                alpha = max(MIN_ALPHA, min(MAX_ALPHA, alpha))

                if "grad_divergence" in metrics:
                    grad_div = float(metrics["grad_divergence"])
                    if not (math.isnan(grad_div) or math.isinf(grad_div)):
                        grad_divergences.append(grad_div)

                local_params = fl.common.parameters_to_ndarrays(parameters)
                if len(local_params) != len(old_global_params):
                    continue
                valid_results.append((local_params, alpha, num_examples, metrics))
            except Exception as e:
                print(f"Error processing client result: {e}")
                continue

        if not valid_results:
            return self.current_global_params, {}

        if ENABLE_ALPHA_FILTERING and len(valid_results) > 1:
            all_alphas = [alpha for _, alpha, _, _ in valid_results]
            alpha_mean = np.mean(all_alphas)
            print(f"[Alpha Filtering] Mean alpha: {alpha_mean:.4f}")
            filtered_results = []
            filtered_out_malicious = 0
            filtered_out_total = 0
            for local_params, alpha, num_examples, metrics in valid_results:
                is_mal = int(metrics.get("is_malicious", 0))
                if alpha >= alpha_mean:
                    filtered_results.append((local_params, alpha, num_examples, metrics))
                else:
                    filtered_out_total += 1
                    if is_mal == 1:
                        filtered_out_malicious += 1
                    print(f"[Alpha Filtering] Filtered client alpha={alpha:.4f} (malicious={bool(is_mal)})")
            print(f"[Alpha Filtering] Filtered {filtered_out_total} clients "
                  f"({filtered_out_malicious} malicious), kept {len(filtered_results)}")
            self.num_filtered_per_round.append(filtered_out_total)
            self.num_filtered_malicious_per_round.append(filtered_out_malicious)
            self.alpha_threshold_per_round.append(alpha_mean)
            if filtered_results:
                valid_results = filtered_results
            else:
                self.num_filtered_per_round[-1] = 0
                self.num_filtered_malicious_per_round[-1] = 0
        else:
            self.num_filtered_per_round.append(0)
            self.num_filtered_malicious_per_round.append(0)
            self.alpha_threshold_per_round.append(0.0)

        mal_alphas_round = []
        benign_alphas_round = []
        for local_params, alpha, num_examples, metrics in valid_results:
            param_diff = [lp - gp for lp, gp in zip(local_params, old_global_params)]
            for i in range(len(weighted_updates)):
                weighted_updates[i] += alpha * param_diff[i]
            sum_alpha += alpha
            num_examples_total += num_examples
            if "residual_loss" in metrics:
                loss_value = float(metrics["residual_loss"])
                if not (math.isnan(loss_value) or math.isinf(loss_value)):
                    residual_losses.append(loss_value)
            alphas.append(alpha)
            if int(metrics.get("is_malicious", 0)) == 1:
                mal_alphas_round.append(alpha)
            else:
                benign_alphas_round.append(alpha)

        if sum_alpha > EPSILON:
            for i in range(len(weighted_updates)):
                weighted_updates[i] = (weighted_updates[i] / sum_alpha) * learning_rate_server
        else:
            weighted_updates = [np.zeros_like(p) for p in old_global_params]

        new_global_params = [gp + weighted_updates[i] for i, gp in enumerate(old_global_params)]
        self.current_global_params = fl.common.ndarrays_to_parameters(new_global_params)

        mean_alpha = np.mean(alphas) if alphas else 1.0
        self.global_ensemble.add_learner(new_global_params, mean_alpha)
        self.global_learners_history.append(new_global_params)
        self.global_alphas_history.append(mean_alpha)

        ensemble_info = self.global_ensemble.get_ensemble_info()
        test_metrics = self.evaluate_ensemble()

        eval_rounds.append(rnd)
        eval_loss_history.append(test_metrics["loss"])
        eval_accuracy_history.append(test_metrics["accuracy"])
        eval_precision_history.append(test_metrics["precision"])
        eval_recall_history.append(test_metrics["recall"])
        eval_f1_history.append(test_metrics["f1_score"])

        print(f"[Round {rnd}] learners={ensemble_info['num_learners']} "
              f"acc={test_metrics['accuracy']:.4f} f1={test_metrics['f1_score']:.4f}")

        mean_residual_loss = np.mean(residual_losses) if residual_losses else 0.0
        mean_grad_div = np.mean(grad_divergences) if grad_divergences else 0.0

        self.mal_alpha_rounds.append(rnd)
        self.mal_alpha_lists.append(mal_alphas_round)
        self.mal_alpha_means.append(float(np.mean(mal_alphas_round)) if mal_alphas_round else np.nan)
        self.benign_alpha_lists.append(benign_alphas_round)
        self.benign_alpha_means.append(float(np.mean(benign_alphas_round)) if benign_alphas_round else np.nan)

        return self.current_global_params, {
            "num_examples": num_examples_total,
            "residual_loss": mean_residual_loss,
            "alpha": mean_alpha,
            "grad_divergence": mean_grad_div,
            "ensemble_size": ensemble_info["num_learners"],
            "mal_alpha_mean_round": self.mal_alpha_means[-1],
            "mal_alpha_count_round": int(len(mal_alphas_round)),
            "benign_alpha_mean_round": self.benign_alpha_means[-1],
            "num_filtered": self.num_filtered_per_round[-1],
            "num_filtered_malicious": self.num_filtered_malicious_per_round[-1],
            "alpha_threshold": self.alpha_threshold_per_round[-1],
            **test_metrics,
        }

    def aggregate_evaluate(self, rnd, results, failures):
        return 0.0, {}

## 10. Experiment runner

In [10]:
def reset_eval_histories():
    global eval_loss_history, eval_accuracy_history, eval_precision_history
    global eval_recall_history, eval_f1_history, eval_grad_divergence_history, eval_rounds
    eval_loss_history = []
    eval_accuracy_history = []
    eval_precision_history = []
    eval_recall_history = []
    eval_f1_history = []
    eval_grad_divergence_history = []
    eval_rounds = []


def set_poisoning(mal_frac, flip_prob, mode="random", seed=123, source_class=0, target_class=1):
    global ENABLE_LABEL_FLIP, MALICIOUS_FRAC, FLIP_PROB, FLIP_MODE
    global SOURCE_CLASS, TARGET_CLASS, POISON_SEED, MALICIOUS_CLIENTS
    POISON_SEED = int(seed)
    ENABLE_LABEL_FLIP = (mal_frac > 0) and (flip_prob > 0)
    MALICIOUS_FRAC = float(mal_frac)
    FLIP_PROB = float(flip_prob)
    FLIP_MODE = str(mode)
    SOURCE_CLASS = int(source_class)
    TARGET_CLASS = int(target_class)
    rng = np.random.RandomState(POISON_SEED)
    num_mal = int(NUM_CLIENTS * MALICIOUS_FRAC)
    if num_mal <= 0:
        MALICIOUS_CLIENTS = set()
    else:
        MALICIOUS_CLIENTS = set(rng.choice(np.arange(NUM_CLIENTS), size=num_mal, replace=False).tolist())
    print(f"[Poison] mal_frac={MALICIOUS_FRAC}, flip_prob={FLIP_PROB}, "
          f"malicious_clients={sorted(MALICIOUS_CLIENTS)}")


def run_one_experiment(num_rounds=15, seed=123):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    reset_eval_histories()

    local_strategy = BoostingStrategy(
        fraction_fit=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
    )
    fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=NUM_CLIENTS,
        config=fl.server.ServerConfig(num_rounds=num_rounds),
        strategy=local_strategy,
        client_resources={"num_cpus": CPUS_PER_CLIENT, "num_gpus": GPU_PER_CLIENT},
    )
    if len(eval_rounds) == 0:
        return None

    final = {
        "final_round": eval_rounds[-1],
        "final_loss": float(eval_loss_history[-1]),
        "final_accuracy": float(eval_accuracy_history[-1]),
        "final_precision": float(eval_precision_history[-1]),
        "final_recall": float(eval_recall_history[-1]),
        "final_f1": float(eval_f1_history[-1]),
        "ensemble_size": int(local_strategy.global_ensemble.get_ensemble_info()["num_learners"]),
        "rounds": list(eval_rounds),
        "acc_curve": list(eval_accuracy_history),
        "loss_curve": list(eval_loss_history),
    }
    mal_means = np.array(local_strategy.mal_alpha_means, dtype=float)
    benign_means = np.array(local_strategy.benign_alpha_means, dtype=float)
    mal_alpha_last_list = local_strategy.mal_alpha_lists[-1] if local_strategy.mal_alpha_lists else []
    benign_alpha_last_list = local_strategy.benign_alpha_lists[-1] if local_strategy.benign_alpha_lists else []
    final.update({
        "mal_alpha_mean_last_round": float(mal_means[~np.isnan(mal_means)][-1]) if np.any(~np.isnan(mal_means)) else np.nan,
        "benign_alpha_mean_last_round": float(benign_means[~np.isnan(benign_means)][-1]) if np.any(~np.isnan(benign_means)) else np.nan,
        "mal_alpha_mean_over_rounds": float(np.nanmean(mal_means)) if np.any(~np.isnan(mal_means)) else np.nan,
        "benign_alpha_mean_over_rounds": float(np.nanmean(benign_means)) if np.any(~np.isnan(benign_means)) else np.nan,
        "mal_alpha_list_last_round": json.dumps([float(a) for a in mal_alpha_last_list]),
        "benign_alpha_list_last_round": json.dumps([float(a) for a in benign_alpha_last_list]),
        "total_filtered": int(sum(local_strategy.num_filtered_per_round)),
        "total_filtered_malicious": int(sum(local_strategy.num_filtered_malicious_per_round)),
        "avg_filtered_per_round": float(np.mean(local_strategy.num_filtered_per_round)),
        "avg_filtered_malicious_per_round": float(np.mean(local_strategy.num_filtered_malicious_per_round)),
    })
    return final

In [11]:

mal_fracs = [0.1,0.3,0.5,0.7]
flip_probs = [1.0]

results = []
curves = {}
for mf in mal_fracs:
    for fp in flip_probs:
        set_poisoning(mal_frac=mf, flip_prob=fp, mode="random", seed=BASE_SEED)
        res = run_one_experiment(num_rounds=NUM_ROUNDS, seed=BASE_SEED)
        if res is None:
            continue
        row = {
            "mode": "random",
            "mal_frac": mf,
            "flip_prob": fp,
            "final_accuracy": res["final_accuracy"],
            "final_f1": res["final_f1"],
            "final_precision": res["final_precision"],
            "final_recall": res["final_recall"],
            "final_loss": res["final_loss"],
            "ensemble_size": res["ensemble_size"],
            "mal_alpha_mean_last_round": res["mal_alpha_mean_last_round"],
            "mal_alpha_mean_over_rounds": res["mal_alpha_mean_over_rounds"],
            "benign_alpha_mean_last_round": res["benign_alpha_mean_last_round"],
            "benign_alpha_mean_over_rounds": res["benign_alpha_mean_over_rounds"],
            "total_filtered": res["total_filtered"],
            "total_filtered_malicious": res["total_filtered_malicious"],
            "avg_filtered_per_round": res["avg_filtered_per_round"],
            "avg_filtered_malicious_per_round": res["avg_filtered_malicious_per_round"],
        }
        results.append(row)
        curves[(mf, fp)] = (res["rounds"], res["acc_curve"])
        print(f"[Sweep] mal_frac={mf:.2f} acc={row['final_accuracy']:.4f} "
              f"filtered_mal={row['total_filtered_malicious']}/{row['total_filtered']}")

df_results = pd.DataFrame(results).sort_values(["mal_frac", "flip_prob"]).reset_index(drop=True)
df_results.to_csv("boostfl_botiot_labelflip.csv", index=False)
df_results

[Poison] mal_frac=0.1, flip_prob=1.0, malicious_clients=[2]


	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=15, no round_timeout
2026-09-18 10:37:06,457	INFO worker.py:1771 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'node:172.24.90.50': 1.0, 'accelerator_type:G': 1.0, 'node:__internal_head__': 1.0, 'CPU': 20.0, 'object_store_memory': 6656582860.0, 'memory': 13313165723.0, 'GPU': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 10, 'num_gpus': 0.5}
INFO :      

[Round 1] 10 clients succeeded, 0 failed
[Alpha Filtering] Mean alpha: 0.9422
[Alpha Filtering] Filtered client alpha=0.8962 (malicious=True)
[Alpha Filtering] Filtered client alpha=0.8872 (malicious=False)
[Alpha Filtering] Filtered client alpha=0.9322 (malicious=False)
[Alpha Filtering] Filtered 3 clients (1 malicious), kept 7


INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
(ClientAppActor pid=877015) 
(ClientAppActor pid=877015)         
(ClientAppActor pid=877015) 
(ClientAppActor pid=877015)         
(ClientAppActor pid=877015) 
(ClientAppActor pid=877015)         
(ClientAppActor pid=877017) 
(ClientAppActor pid=877017)         
(ClientAppActor pid=877017) 
(ClientAppActor pid=877017)         
(ClientAppActor pid=877017) 
(ClientAppActor pid=877017)         
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 2]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Round 1] learners=1 acc=0.5249 f1=0.3574


...

INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
(ClientAppActor pid=883215) 
(ClientAppActor pid=883215)         
(ClientAppActor pid=883215) 
(ClientAppActor pid=883215)         
(ClientAppActor pid=883215) 
(ClientAppActor pid=883215)         
(ClientAppActor pid=883215) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context` [repeated 20x across cluster]
(ClientAppActor pid=883215)             This is a deprecated feature. It will be removed [repeated 20x across cluster]
(ClientAppActor pid=883215)             entirely in future versions of Flower. [repeated 20x across cluster]
(ClientAppActor pid=883218) 
(ClientAppActor pid=883218)         
(ClientAppActor pid=883218) 
(ClientAppActor pid=883218)         
(ClientAppActor pid=883218) 
(ClientAppActor pid=883218)         
(C

[Round 15] learners=15 acc=0.9709 f1=0.9592


INFO :      	            (7, 0.9761083328196805),
INFO :      	            (8, 0.9768543451610474),
INFO :      	            (9, 0.9768503575024141),
INFO :      	            (10, 0.976842382185148),
INFO :      	            (11, 0.9770130365104477),
INFO :      	            (12, 0.9770130365104477),
INFO :      	            (13, 0.9771876784943807),
INFO :      	            (14, 0.9771876784943807),
INFO :      	            (15, 0.977646011827714)],
INFO :      	 'residual_loss': [(1, np.float64(0.05273136662771107)),
INFO :      	                   (2, np.float64(0.03477355929571107)),
INFO :      	                   (3, np.float64(0.03272707949524457)),
INFO :      	                   (4, np.float64(0.03232619570847207)),
INFO :      	                   (5, np.float64(0.032170281746626374)),
INFO :      	                   (6, np.float64(0.03209735748521516)),
INFO :      	                   (7, np.float64(0.032034559293807274)),
INFO :      	                   (8, np.float64(0.0320

[Sweep] mal_frac=0.70 acc=0.9709 filtered_mal=105/105


,mode,mal_frac,flip_prob,final_accuracy,final_f1,final_precision,final_recall,final_loss,ensemble_size,mal_alpha_mean_last_round,mal_alpha_mean_over_rounds,benign_alpha_mean_last_round,benign_alpha_mean_over_rounds,total_filtered,total_filtered_malicious,avg_filtered_per_round,avg_filtered_malicious_per_round
0,random,0.1,1.0,0.973792,0.963488,0.971152,0.957855,0.229210,15,NaN,NaN,0.968288,0.967220,22,15,1.466667,1.0
1,random,0.3,1.0,0.991069,0.984496,0.980515,0.988969,0.095263,15,NaN,NaN,0.968949,0.967617,46,45,3.066667,3.0
2,random,0.5,1.0,0.987418,0.974773,0.966144,0.984992,0.100002,15,NaN,NaN,0.969011,0.967368,75,75,5.000000,5.0
3,random,0.7,1.0,0.970924,0.959159,0.943603,0.977646,0.110871,15,NaN,NaN,0.969032,0.967476,105,105,7.000000,7.0
